In [1]:
## In this file, we load the required libraries, retrieve Spotify API credentials from the .env file,
## and verify that both credentials are successfully loaded.

import os, time, requests
import pandas as pd
from dotenv import load_dotenv

# Load API credentials
load_dotenv("../.env")
CID = os.getenv("SPOTIFY_CLIENT_ID")
SECRET = os.getenv("SPOTIFY_CLIENT_SECRET")

# Confirm both values were found
print("ID loaded:", CID is not None, "| Secret loaded:", SECRET is not None)

ID loaded: True | Secret loaded: True


In [2]:
## We request a Spotify API access token using the Client Credentials flow, verify the request was successful,
## and set up the authorization headers and base API URL.

# Request an access token using the Client Credentials flow
r = requests.post("https://accounts.spotify.com/api/token",
                  data={"grant_type": "client_credentials"},
                  auth=(CID, SECRET))
print(r.status_code)
r.raise_for_status()
HEADERS = {"Authorization": f"Bearer {r.json()['access_token']}"}
BASE = "https://api.spotify.com/v1"
print("Token OK")

200
Token OK


In [3]:
## Search Spotify for The Weeknd, display the top five artist matches, and select the first match.

# Search Spotify for the artist by name
ARTIST = "The Weeknd"

r = requests.get(
    f"{BASE}/search",          
    headers=HEADERS,           
    params={"q": ARTIST,  # search keyword
            "type": "artist",  
            "limit": 5}  # return top 5 matches to verify
)
r.raise_for_status()

# Show the top matches so we can confirm
results = r.json()["artists"]["items"]
for a in results:
    print(a["name"], "|", a["id"])

# Take the first (best) match
artist_id = results[0]["id"]
print("\nUsing:", results[0]["name"], artist_id)

The Weeknd | 1Xyo4u8uXC1ZmMpatF05PJ
The Weeks | 3zE5jV6Uw9hhdWCXM8hS3j
SZA | 7tYKF4w9nC0nq9CsPZTHyP
Drake | 3TVXtAsR1Inumwj472S9r4
Weekend | 3Z73qbDrey1ubsAofhOoz6

Using: The Weeknd 1Xyo4u8uXC1ZmMpatF05PJ


In [4]:
## Retrieve all full albums by the selected artist, extract key album information,
## and organize the results into a DataFrame.

# Get all albums by the artist
albums = []
url = f"{BASE}/artists/{artist_id}/albums"
params = {"include_groups": "album",  # only full albums
          "limit": 10}

while url:
    r = requests.get(url, headers=HEADERS, params=params)
    r.raise_for_status()
    j = r.json()
    albums += j["items"]  # add this page's albums to our list
    url = j["next"]                    
    params = None                      
    time.sleep(0.2)

# Put the key album info into a DataFrame
albums_df = pd.DataFrame([{
    "album_id": a["id"],
    "album": a["name"],
    "release_date": a["release_date"],
    "total_tracks": a["total_tracks"]
} for a in albums]).sort_values("release_date")

albums_df

,album_id,album,release_date,total_tracks
13,7zCODUHkfuRxsUjtuzNqbd,House Of Balloons (Original),2011-03-21,9
12,6F87lH0I09qlrzvCCKc7lz,Thursday (Original),2011-08-18,9
11,04hy4jb1GDD00otiwzsFUB,Echoes Of Silence (Original),2011-12-21,9
10,2FgMWuwMeTgJArP2RF3upF,Kiss Land,2013-01-01,10
9,0P3oVJBFOv3TDXlYRhGL7s,Beauty Behind The Madness,2015-08-28,14
8,2ODvWsOgouMbaA5xf0RkJe,Starboy,2016-11-25,18
7,4qZBW3f2Q8y0k1A84d4iAO,"My Dear Melancholy,",2018-03-30,7
5,4yP0hdKOZPNshxUOjY0cZj,After Hours,2020-03-20,14
6,742eAldb4AJKLoPgJhGRE7,After Hours (Deluxe),2020-03-20,18
4,2nLOHgzXzwFEpl62zAgCEC,Dawn FM,2022-01-06,16


In [5]:
## Filter the DataFrame to include only six standard studio albums and remove duplicates.

# Keep only the 6 standard studio albums
STUDIO_ALBUMS = ["Kiss Land", "Beauty Behind The Madness", "Starboy",
                 "After Hours", "Dawn FM", "Hurry Up Tomorrow"]

# Exact name match
mask = albums_df["album"].str.lower().isin([n.lower() for n in STUDIO_ALBUMS])
selected = (albums_df[mask]
            .drop_duplicates(subset="album", keep="first")
            .reset_index(drop=True))
selected

,album_id,album,release_date,total_tracks
0,2FgMWuwMeTgJArP2RF3upF,Kiss Land,2013-01-01,10
1,0P3oVJBFOv3TDXlYRhGL7s,Beauty Behind The Madness,2015-08-28,14
2,2ODvWsOgouMbaA5xf0RkJe,Starboy,2016-11-25,18
3,4yP0hdKOZPNshxUOjY0cZj,After Hours,2020-03-20,14
4,2nLOHgzXzwFEpl62zAgCEC,Dawn FM,2022-01-06,16
5,3OxfaVgvTxUTy7276t7SPU,Hurry Up Tomorrow,2025-01-31,22


In [6]:
## Retrieve all tracks from the six studio albums, extract key track information,
## and organize the results into a DataFrame.

# Loop through each album and get its tracks
rows = []
for _, a in selected.iterrows():
    url = f"{BASE}/albums/{a['album_id']}/tracks"
    params = {"limit": 50}  # max tracks per page
    while url:
        r = requests.get(url, headers=HEADERS, params=params)
        r.raise_for_status()
        j = r.json()
        for t in j["items"]:
            rows.append({
                "Album": a["album"],
                "Song Name": t["name"],
                "Track Number": t["track_number"],
                "Duration (s)": round(t["duration_ms"] / 1000, 1),  # ms -> s
                
                # additional fields
                "Explicit": t["explicit"],  # whether the track has explicit content
                "Release Date": a["release_date"],  # album release date
                "Track ID": t["id"],  # Spotify's unique track ID
            })
        url = j["next"]
        params = None
    time.sleep(0.2)

spotify_df = pd.DataFrame(rows)
print(spotify_df.shape)
spotify_df.head(10)

(94, 7)


,Album,Song Name,Track Number,Duration (s),Explicit,Release Date,Track ID
0,Kiss Land,Professional,1,368.7,False,2013-01-01,2gOEWbtEzHtpHb9fywSEiw
1,Kiss Land,The Town,2,307.3,False,2013-01-01,3RB6zxrgPQBUxpTtoMW60r
2,Kiss Land,Adaptation,3,283.9,False,2013-01-01,31ue6kAOfrqVrdoPdfkq40
3,Kiss Land,Love In The Sky,4,267.9,True,2013-01-01,7fSKE9J3udo8zpq1ceYTEa
4,Kiss Land,Belong To The World,5,307.2,False,2013-01-01,2jst5B18Q3PN3rwwIfnAgp
5,Kiss Land,Live For,6,222.7,True,2013-01-01,5TiqpChMv5rqzAwfRr8h1Y
6,Kiss Land,Wanderlust,7,306.8,False,2013-01-01,3b30Kdm3nKrD352zEPCAda
7,Kiss Land,Kiss Land,8,455.3,True,2013-01-01,634tM1Us7Z6bsbZhRbrRr2
8,Kiss Land,Pretty,9,375.4,True,2013-01-01,1j0TqbB4ajwaJqyZnEK8kO
9,Kiss Land,Tears In The Rain,10,444.7,False,2013-01-01,4TgYcx5TPQF7xC79b6v4hP


In [7]:
# Check track counts per album
spotify_df["Album"].value_counts()

Album
Hurry Up Tomorrow            22
Starboy                      18
Dawn FM                      16
Beauty Behind The Madness    14
After Hours                  14
Kiss Land                    10
Name: count, dtype: int64

In [8]:
# Save the Spotify data to the data folder for use in Part 3
spotify_df.to_csv("../data/spotify_tracks.csv", index=False)
print("Saved to data/spotify_tracks.csv")

Saved to data/spotify_tracks.csv
